# 🩺 WellAware AI – Run on Google Colab
**Patient Wellness Campaign Visual Generator · Team 17**

Before you start: **Runtime → Change runtime type → T4 GPU → Save**.

Run the cells from top to bottom. The last cell prints a public link that opens the Streamlit app.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Upload and unzip the project
Run the cell and choose **WellAware-AI.zip** from your computer.

*(Alternative: put the zip in Google Drive, mount it with `from google.colab import drive; drive.mount('/content/drive')`, and copy it to `/content`.)*

In [ ]:
%cd /content
from google.colab import files  # type: ignore
uploaded = files.upload()          # pick WellAware-AI.zip
!unzip -q -o WellAware-AI.zip -d /content
%cd /content/WellAware-AI
!ls

## 3. Install dependencies (~1 minute)

In [ ]:
%pip install -q -r requirements.txt

## 4. (Optional) Hugging Face token
Not required — both models are public. A token only avoids download rate limits.
Add it in Colab's 🔑 **Secrets** panel as `HF_TOKEN`, then run this cell.

In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("✅ HF_TOKEN loaded")
    else:
        print("ℹ️ No HF_TOKEN secret found – continuing without it (that's fine).")
except Exception:
    print("ℹ️ No HF_TOKEN secret found – continuing without it (that's fine).")

## 5. Quick test – downloads the model and generates one image

In [ ]:
!python scripts/quick_test.py
import os
from IPython.display import Image, display
if os.path.exists("generated_images/quick_test.png"):
    display(Image("generated_images/quick_test.png"))
else:
    print("Quick test did not produce an image. Fix the error above first.")

## 6. Start the Streamlit app in the background

In [ ]:
!pkill -f "streamlit run" || true
!nohup streamlit run app.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &
import time
for _ in range(30):
    import urllib.request
    try:
        if urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).read() == b"ok":
            break
    except Exception:
        pass
    time.sleep(1)
!tail -n 5 /content/streamlit.log

## 7. Get a public link (Cloudflare tunnel – no account needed)
Open the printed `https://….trycloudflare.com` link. Keep this notebook running while you use the app.

In [ ]:
!test -f cloudflared || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared)
import subprocess, re
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("🌐 Open WellAware AI here:", match.group(0))
        break

### Alternative tunnel: localtunnel
Use this only if Cloudflare doesn't work. The page will ask for a password — it is the IP address printed by the first line.

In [ ]:
!curl -s https://ipv4.icanhazip.com
!npx --yes localtunnel --port 8501

## 8. Troubleshooting
- **App log:** `!tail -n 40 /content/streamlit.log`
- **CUDA out of memory:** use 512 × 512 and 1 variation, or *Runtime → Restart session* and run cells 6–7 again.
- **Link stopped working:** Colab disconnected — rerun cells 6 and 7.
- **Download generated images:** use the app's *Download* buttons or the Gallery's *Download all (ZIP)*.

In [ ]:
!tail -n 40 /content/streamlit.log